# IMS Sensor Processing Project — Hydrophone Analysis
**Arian Dannemann, Bashar Haidar, Juri Schallück**
**University of Bremen — Intelligent Marine Systems, June 2026**

---

## Overview

This notebook presents the acoustic analysis of hydrophone recordings collected at the Unisee (Stadtwaldsee) in Bremen on 13.05.2026. An Ocean Sonics icListen HF SJ9-ETH hydrophone was deployed adjacent to a small pier, recording continuously from 08:30 to 15:00 UTC at 32,000 Hz.

The analysis follows two parallel tracks:

1. **General acoustic analysis** — statistics, denoising, and spectrogram inspection across all 40 recorded files, identifying Otter USV activity, duck events, and other acoustic signatures.
2. **Automated rainfall detection** — a signal processing pipeline that detects rain onset from spectral features, complementing the manual inspection with quantitative timestamps and confidence scores.

### Logged Events (Ground Truth)

| Time (UTC) | Event |
|---|---|
| 08:30 | Hydrophone activated |
| 08:31 | Hydrophone deployed in water |
| 09:32 | Ducks near hydrophone |
| 10:07 | Ducks near hydrophone |
| 10:37 | Otter USV |
| 10:51 | Hydrophone removed for data check |
| 11:11 | Hydrophone back in water |
| 11:30 | **Rain onset** |
| 12:45 | Otter USV |
| 14:17 | Otter USV |
| 14:53 | Otter USV |


## 0. Environment

In [ ]:
# Install dependencies if needed:
# !pip install numpy scipy matplotlib librosa soundfile noisereduce pandas

import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import librosa
import noisereduce as nr
from scipy.signal import butter, sosfiltfilt
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

print(f"numpy      {np.__version__}")
print(f"pandas     {pd.__version__}")
print(f"librosa    {librosa.__version__}")
print(f"soundfile  {sf.__version__}")


## 1. Configuration

All analysis parameters are defined here for reproducibility.


In [ ]:
# ── Data ─────────────────────────────────────────────────────────────────────
DATA_DIR   = "./data"
OUTPUT_CSV = "./rain_detections.csv"

# ── Blacklist ─────────────────────────────────────────────────────────────────
# Files excluded from analysis:
# {0, 1}      — hydrophone being placed in water (deployment artefacts)
# {14, 15, 16} — hydrophone removed for data check (10:51–11:11 UTC)
# {37, 38, 39} — Otter USV extremely close, signal saturated
BLACKLIST = {0, 1, 14, 15, 16, 37, 38, 39}

# ── Denoising (Section 2.2) ───────────────────────────────────────────────────
NOISE_FRACTION = 0.10   # first 10% of each file used as noise profile
AMP_FACTOR     = 5      # amplification after denoising

# ── STFT ─────────────────────────────────────────────────────────────────────
N_FFT      = 2048        # frequency resolution = 32000 / 2048 ≈ 15.6 Hz/bin
HOP_LENGTH = 512         # time resolution      = 512 / 32000  ≈ 16 ms/frame

# ── Rainfall Detection (Section 2.4) ─────────────────────────────────────────
RAIN_BAND        = (5_000, 16_000)   # Hz — diagnostic HF band
LF_BAND          = (200,   2_000)    # Hz — Otter USV motor band
THRESHOLD_K      = 0.2               # threshold = μ + K·σ
CLIP_PCT         = 60                # percentile for spike clipping
MIN_ENERGY_DB    = -65.0             # absolute minimum energy floor
SMOOTH_S         = 5.0               # energy smoothing window (seconds)
MIN_RAIN_S       = 20.0              # minimum duration for a valid rain segment
MERGE_GAP_S      = 15.0              # merge gaps shorter than this
SPIKE_RISE_DB    = 3.0               # max allowed energy rise rate (dB/s)
SPIKE_MAX_RATIO  = 0.4               # max fraction of steep frames per segment
RATIO_MAX        = 4.0               # max LF−HF difference (dB)

print("Configuration loaded.")
print(f"  Blacklist      : {sorted(BLACKLIST)}")
print(f"  Rain band      : {RAIN_BAND[0]:,}–{RAIN_BAND[1]:,} Hz")
print(f"  Threshold      : μ + {THRESHOLD_K}·σ  (Clip-Median @ {CLIP_PCT}th pct)")
print(f"  Min duration   : {MIN_RAIN_S} s")


## 2. Methodology

### 2.1 Data Loading

Files are read using `soundfile` and converted to mono float32.


In [ ]:
def get_signal(path):
    """Read a .wav file and return (signal, sample_rate) as mono float32."""
    signal, sample_rate = sf.read(path, dtype="float32", always_2d=True)
    return signal.mean(axis=1), sample_rate


def parse_utc(fname):
    """Extract UTC start time from filename: SJW7447_20260513_083000.wav"""
    b = os.path.basename(fname).replace(".wav", "").split("_")
    return datetime.strptime(f"{b[1]}_{b[2]}", "%Y%m%d_%H%M%S")


wav_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.wav")))
if not wav_files:
    raise FileNotFoundError(f"No .wav files found in {DATA_DIR!r}")

print(f"Found {len(wav_files)} files.")
print(f"First : {os.path.basename(wav_files[0])}")
print(f"Last  : {os.path.basename(wav_files[-1])}")


### 2.2 Basic Statistics

Following the group report methodology, we compute mean amplitude, maximum amplitude,
and variance for each file. This provides a first overview and helps identify
files dominated by anomalous events.


In [ ]:
stats = []
for i, path in enumerate(wav_files):
    y, sr = get_signal(path)
    a     = np.abs(y)
    stats.append({
        "idx" : i,
        "file": os.path.basename(path),
        "utc" : parse_utc(path).strftime("%H:%M"),
        "mean": float(np.mean(a)),
        "max" : float(np.max(a)),
        "var" : float(np.var(a)),
    })

stats_df = pd.DataFrame(stats)
print(f"Statistics computed for {len(stats_df)} files.")


In [ ]:
# Figure 1: Statistics for all 40 files
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
fig.suptitle("Figure 1: Mean, Maximum and Variance — all 40 recordings",
             fontsize=12, fontweight="bold")

for ax, col, lbl in zip(axes,
                         ["mean", "max", "var"],
                         ["Mean Amplitude", "Max Amplitude", "Variance"]):
    ax.plot(stats_df["idx"], stats_df[col], "o-", ms=4, lw=1.2, color="#2c7bb6")
    ax.set_ylabel(lbl, fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Recording Index")
plt.tight_layout()
plt.show()


In [ ]:
# Figure 2: Filtered statistics (blacklist removed)
filt = stats_df[~stats_df["idx"].isin(BLACKLIST)].reset_index(drop=True)

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
fig.suptitle("Figure 2: Statistics after blacklist removal (brown=Ducks, blue=Otter, green=Rain)",
             fontsize=11, fontweight="bold")

for ax, col, lbl in zip(axes,
                         ["mean", "max", "var"],
                         ["Mean Amplitude", "Max Amplitude", "Variance"]):
    ax.plot(range(len(filt)), filt[col], "o-", ms=4, lw=1.2, color="#2c7bb6")
    ax.set_ylabel(lbl, fontsize=9)
    ax.grid(True, alpha=0.3)

    for j, row in filt.iterrows():
        hm = row["utc"]
        if hm in ["09:30", "10:10"]:
            ax.axvline(j, color="saddlebrown", ls="--", lw=1.2, alpha=0.6)
        if hm in ["10:40", "12:40", "14:10", "14:50"]:
            ax.axvline(j, color="steelblue",   ls="--", lw=1.2, alpha=0.6)
        if hm == "11:30":
            ax.axvline(j, color="green", ls="--", lw=2.0)

axes[-1].set_xlabel("Filtered Index")
plt.tight_layout()
plt.show()


### 2.3 Noise Reduction

Noise reduction is applied using `noisereduce` with the first 10% of each file
as the noise profile, followed by a tanh compressor and amplification ×5.
This makes quiet events audible and improves spectrogram contrast.

The denoised signal is used for **visualization and listening** only.
See Section 2.4 for the rationale behind using the raw signal for detection.


In [ ]:
def get_denoised_signal(path, amp=AMP_FACTOR):
    """Denoise a recording using spectral gating + tanh compression.

    Noise profile: first 10% of the file.
    prop_decrease=0.5 (reduced from 0.999) to preserve broadband rain signature
    for visualization purposes.
    """
    audio_data, sample_rate = get_signal(path)
    noise_lim    = int(len(audio_data) * NOISE_FRACTION)
    noise_sample = audio_data[:noise_lim]

    reduced = nr.reduce_noise(
        y=audio_data, sr=sample_rate,
        y_noise=noise_sample,
        prop_decrease=0.5,
        stationary=True,
        n_std_thresh_stationary=2.0
    )

    # tanh compressor: reduces extreme peaks
    compressed = np.tanh(reduced * 1.5)
    peak = np.max(np.abs(compressed))
    if peak > 0:
        reduced = (compressed / peak) * 1.0

    return (reduced * amp).astype(np.float32), sample_rate


# Example: raw vs denoised comparison
example_path = None
for p in wav_files:
    if parse_utc(p).strftime("%H:%M") == "11:30":
        example_path = p
        break
if not example_path:
    example_path = wav_files[20]

y_raw, sr = get_signal(example_path)
y_dn,  _  = get_denoised_signal(example_path)

fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
t = np.arange(len(y_raw)) / sr
axes[0].plot(t, y_raw, lw=0.3, color="#888")
axes[0].set(title=f"Figure 3a: Raw signal — {os.path.basename(example_path)}",
            ylabel="Amplitude")
axes[1].plot(t, y_dn,  lw=0.3, color="#2c7bb6")
axes[1].set(title=f"Figure 3b: Denoised signal (x{AMP_FACTOR}) — {os.path.basename(example_path)}",
            ylabel="Amplitude", xlabel="Time (s)")
plt.tight_layout()
plt.show()


### 2.4 Rainfall Detection

Building on the denoised spectrograms produced in Section 2.3, an automated
detector was developed to identify sustained broadband high-frequency energy
consistent with the acoustic signature of rainfall.

When rain drops impact the water surface, they entrap small air bubbles that
resonate acoustically, producing broadband noise in the 5,000–16,000 Hz range
(Nystuen & Medwin, 1993). At our Nyquist limit of 16,000 Hz, we capture the
lower edge of this resonance band.

The detection operates on the **raw signal** (20 Hz high-pass only). Using the
fully denoised signal would suppress the rain signature, because broadband
stationary noise — the acoustic character of rain — is exactly what spectral
gating removes.

#### Detection Pipeline

```
Raw signal → 20 Hz HP → STFT → HF band energy (5–16 kHz)
                                        ↓
                            Clip-Median threshold
                                        ↓
                         ① Spike filter     (rejects Otter impulses)
                         ② LF/HF ratio      (rejects Otter motor noise)
                         ③ Min duration     (rejects duck splashes)
                                        ↓
                         Confirmed rain segment + p(rain)
```


In [ ]:
def highpass_20hz(y, sr):
    """20 Hz high-pass filter — removes DC drift only, preserves rain signal."""
    sos = butter(4, 20.0, btype="highpass", fs=sr, output="sos")
    return sosfiltfilt(sos, y).astype(np.float32)


def make_spectrogram(y, sr):
    """Compute STFT spectrogram in dB scale."""
    S    = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH, window="hann"))
    S_db = librosa.amplitude_to_db(S, ref=np.max)
    fr   = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)
    tm   = librosa.frames_to_time(np.arange(S_db.shape[1]), sr=sr, hop_length=HOP_LENGTH)
    return S_db, fr, tm


def auto_contrast_specshow(ax, times, freqs, S_db, cmap="magma"):
    """Plot spectrogram with percentile-based contrast (avoids uniform-color issue)."""
    vmin = np.percentile(S_db, 2)
    vmax = np.percentile(S_db, 98)
    img  = ax.pcolormesh(times, freqs, S_db, shading="auto",
                         cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_ylim(0, 16000)
    return img, vmin, vmax


def compute_band_energy(S_db, freqs, times, band_hz, smooth_s):
    """Mean energy within a frequency band, smoothed over a time window."""
    lo, hi = band_hz
    mask   = (freqs >= lo) & (freqs <= hi)
    energy = np.mean(S_db[mask, :], axis=0)
    dt     = times[1] - times[0]
    k      = max(int(round(smooth_s / dt)), 1)
    return np.convolve(energy, np.ones(k) / k, mode="same")


def detect_rain(energy, lf_energy, times):
    """Clip-Median threshold + three-stage filter → confirmed rain segments.

    Returns (threshold, mu, sigma, clip_level, segments, rejected)
    """
    dt = times[1] - times[0]

    # Clip-Median: robust against Otter USV spike bias
    clip_level = np.percentile(energy, CLIP_PCT)
    e_clip     = np.clip(energy, None, clip_level)
    mu         = np.median(e_clip)
    sigma      = np.median(np.abs(e_clip - mu)) * 1.4826   # MAD → std
    threshold  = max(mu + THRESHOLD_K * sigma, MIN_ENERGY_DB)

    above = energy > threshold

    # Find raw segments
    segs, in_s, s0 = [], False, 0
    for i, v in enumerate(above):
        if v and not in_s:    in_s, s0 = True, i
        elif not v and in_s:  in_s = False; segs.append((s0, i - 1))
    if in_s: segs.append((s0, len(above) - 1))

    # Merge short gaps
    mg = max(int(round(MERGE_GAP_S / dt)), 1)
    merged = []
    for s, e in segs:
        if merged and (s - merged[-1][1]) <= mg:
            merged[-1] = (merged[-1][0], e)
        else:
            merged.append([s, e])

    # Filter and score
    min_f    = max(int(round(MIN_RAIN_S / dt)), 1)
    segments, rejected = [], []

    for s, e in merged:
        if (e - s + 1) < min_f:
            rejected.append((s, e, "too short")); continue

        seg_e  = energy[s:e + 1]
        seg_lf = lf_energy[s:e + 1]

        # Filter 1: spike rejection
        diff        = np.diff(seg_e)
        steep       = np.abs(diff) > (SPIKE_RISE_DB * dt)
        steep_ratio = steep.sum() / max(len(diff), 1)
        if steep_ratio > SPIKE_MAX_RATIO:
            rejected.append((s, e, f"spike ({steep_ratio:.2f})")); continue

        # Filter 2: LF/HF ratio
        lf_excess = float(np.mean(seg_lf)) - float(np.mean(seg_e))
        if lf_excess > RATIO_MAX:
            rejected.append((s, e, f"LF-dominated ({lf_excess:.1f} dB)")); continue

        # Probability score
        z    = (float(np.mean(seg_e)) - mu) / (sigma + 1e-9)
        prob = round(float(1.0 / (1.0 + np.exp(-(z - THRESHOLD_K)))), 3)

        segments.append({
            "start_s":    float(times[s]),
            "end_s":      float(times[e]),
            "duration_s": round(float(times[e] - times[s]), 1),
            "mean_db":    round(float(np.mean(seg_e)), 2),
            "lf_db":      round(float(np.mean(seg_lf)), 2),
            "prob":       prob,
        })

    return threshold, mu, sigma, clip_level, segments, rejected


print("Detection functions loaded.")


## 3. Results

### 3.1 Spectrogram Analysis

The following plots show the spectrogram, band energies, and detection output
for the files surrounding the logged rain onset (11:20–11:40 UTC).


In [ ]:
def plot_file(path, make_plot=True):
    """Full 5-panel analysis plot for a single file."""
    name      = os.path.basename(path)
    utc_start = parse_utc(path)

    y, sr     = get_signal(path)
    y_hp      = highpass_20hz(y, sr)
    y_dn, _   = get_denoised_signal(path)

    S_db,  freqs, times = make_spectrogram(y_hp, sr)
    S_dn,  _,     _     = make_spectrogram(highpass_20hz(y_dn, sr), sr)

    energy    = compute_band_energy(S_db, freqs, times, RAIN_BAND, SMOOTH_S)
    lf_energy = compute_band_energy(S_db, freqs, times, LF_BAND,   SMOOTH_S)

    thr, mu, sigma, clip_lv, segments, rejected = detect_rain(energy, lf_energy, times)

    # UTC timestamps
    records = []
    for seg in segments:
        utc_s = (utc_start + timedelta(seconds=seg["start_s"])).strftime("%H:%M:%S")
        utc_e = (utc_start + timedelta(seconds=seg["end_s"]  )).strftime("%H:%M:%S")
        records.append({
            "file_name":        name,
            "utc_start":        utc_s,
            "utc_end":          utc_e,
            "duration_s":       seg["duration_s"],
            "mean_energy_db":   seg["mean_db"],
            "lf_energy_db":     seg["lf_db"],
            "rain_probability": seg["prob"],
        })

    if not make_plot:
        return records

    fig, axes = plt.subplots(5, 1, figsize=(14, 16))
    fig.suptitle(f"{name}  [{utc_start.strftime('%H:%M UTC')}]",
                 fontsize=11, fontweight="bold")

    # Panel 1: Raw signal
    t_ax = np.arange(len(y_hp)) / sr
    axes[0].plot(t_ax, y_hp, lw=0.25, color="#555")
    for seg in segments:
        axes[0].axvspan(seg["start_s"], seg["end_s"], color="red",    alpha=0.18)
    for s, e, _ in rejected:
        axes[0].axvspan(times[s], times[e],          color="orange", alpha=0.15)
    axes[0].set(title="Raw signal (20 Hz HP)  —  red = rain detected, orange = rejected",
                ylabel="Amplitude", xlim=(0, t_ax[-1]))
    axes[0].grid(True, alpha=0.2)

    # Panel 2: Spectrogram RAW
    img1, v1, v2 = auto_contrast_specshow(axes[1], times, freqs, S_db)
    plt.colorbar(img1, ax=axes[1], label="dB", shrink=0.8)
    axes[1].axhline(RAIN_BAND[0], color="cyan",   ls="--", lw=1.2, label=f"HF band {RAIN_BAND[0]:,} Hz")
    axes[1].axhline(RAIN_BAND[1], color="cyan",   ls="--", lw=1.2)
    axes[1].axhline(LF_BAND[1],   color="yellow", ls=":",  lw=1.0, label=f"LF band {LF_BAND[1]:,} Hz")
    for seg in segments:
        axes[1].axvspan(seg["start_s"], seg["end_s"], color="red",    alpha=0.12)
    axes[1].set(title=f"Spectrogram RAW — auto contrast [{v1:.1f}…{v2:.1f} dB]",
                ylabel="Frequency (Hz)")
    axes[1].legend(fontsize=7, loc="upper right")

    # Panel 3: Spectrogram Denoised
    img2, v3, v4 = auto_contrast_specshow(axes[2], times, freqs, S_dn)
    plt.colorbar(img2, ax=axes[2], label="dB", shrink=0.8)
    axes[2].axhline(RAIN_BAND[0], color="cyan",   ls="--", lw=1.2)
    axes[2].axhline(RAIN_BAND[1], color="cyan",   ls="--", lw=1.2)
    axes[2].axhline(LF_BAND[1],   color="yellow", ls=":",  lw=1.0)
    for seg in segments:
        axes[2].axvspan(seg["start_s"], seg["end_s"], color="red",    alpha=0.12)
    axes[2].set(title=f"Spectrogram Denoised (prop=0.5) [{v3:.1f}…{v4:.1f} dB]",
                ylabel="Frequency (Hz)")

    # Panel 4: LF vs HF energy
    axes[3].plot(times, lf_energy, color="#d62728", lw=1.0, alpha=0.8,
                 label=f"LF {LF_BAND[0]}–{LF_BAND[1]} Hz  (Otter indicator)")
    axes[3].plot(times, energy,    color="#2ca02c", lw=1.2,
                 label=f"HF {RAIN_BAND[0]//1000}–{RAIN_BAND[1]//1000} kHz  (Rain indicator)")
    axes[3].axhline(thr, color="red",  ls="--", lw=1.5,
                    label=f"Threshold = {thr:.1f} dB")
    axes[3].axhline(mu,  color="gray", ls=":",  lw=1.0, alpha=0.8,
                    label=f"Median μ = {mu:.1f} dB")
    for seg in segments:
        axes[3].axvspan(seg["start_s"], seg["end_s"], color="red", alpha=0.18)
        axes[3].text(seg["start_s"] + 0.5,
                     thr + (np.max(energy) - thr) * 0.1,
                     f"p={seg['prob']}", color="darkred",
                     fontsize=9, fontweight="bold")
    for s, e, reason in rejected:
        axes[3].axvspan(times[s], times[e], color="orange", alpha=0.15)
    axes[3].set(title="LF vs HF band energy", ylabel="Energy (dB)")
    axes[3].legend(fontsize=8, loc="lower right")
    axes[3].grid(True, alpha=0.2)

    # Panel 5: LF-HF difference
    diff_lf_hf = lf_energy - energy
    axes[4].fill_between(times, diff_lf_hf, 0,
                         where=(diff_lf_hf > RATIO_MAX),
                         color="#d62728", alpha=0.55,
                         label=f"LF > HF + {RATIO_MAX} dB  (Otter-dominated)")
    axes[4].fill_between(times, diff_lf_hf, 0,
                         where=(diff_lf_hf <= RATIO_MAX),
                         color="#2ca02c", alpha=0.45,
                         label="LF ≈ HF  (broadband — rain candidate)")
    axes[4].axhline(RATIO_MAX, color="red", ls="--", lw=1.5,
                    label=f"Otter boundary = {RATIO_MAX} dB")
    axes[4].axhline(0, color="gray", ls=":", lw=1.0)
    axes[4].set(title="LF-HF Difference  (red = Otter-dominated, green = rain candidate)",
                xlabel="Time (s)", ylabel="LF - HF (dB)")
    axes[4].legend(fontsize=8)
    axes[4].grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

    for r in records:
        print(f"  Rain: {r['utc_start']} – {r['utc_end']}  "
              f"| {r['duration_s']:.0f}s  | p={r['rain_probability']}")
    if not records:
        print("  No rain detected.")

    return records


In [ ]:
# Plot files around the rain onset
for path in wav_files:
    if parse_utc(path).strftime("%H:%M") in ["11:20", "11:30", "11:40", "12:00"]:
        print(f"\n{'='*65}")
        _ = plot_file(path, make_plot=True)


### 3.2 Full Dataset — Automated Rain Detection

The detector is applied to all non-blacklisted files. Results are compared
against the ground-truth event log to validate detection accuracy.


In [ ]:
print(f"Processing {len(wav_files) - len(BLACKLIST)} files...")
print("=" * 65)

all_records = []
for i, path in enumerate(wav_files):
    if i in BLACKLIST:
        print(f"  [{i:02d}] SKIP (blacklist)")
        continue

    utc  = parse_utc(path)
    recs = plot_file(path, make_plot=False)
    all_records.extend(recs)

    mark = f"{len(recs)} segment(s)" if recs else "—"
    print(f"  [{i:02d}] {utc.strftime('%H:%M')} UTC  |  {mark}")

print("=" * 65)
print(f"Total: {len(all_records)} rain segments detected")


In [ ]:
df = pd.DataFrame(all_records)

if df.empty:
    print("No rain segments detected.")
else:
    print(f"{len(df)} rain segments across {df['file_name'].nunique()} files\n")
    print(df.to_string(index=False))

    df["after_rain"] = df["utc_start"] >= "11:30"
    before = df[~df["after_rain"]]
    after  = df[ df["after_rain"]]

    print("\n" + "━" * 58)
    print(f"Before 11:30 UTC : {len(before):3d} segment(s)")
    print(f"From   11:30 UTC : {len(after):3d} segment(s)  <- expected rain")
    print("━" * 58)
    if len(after) > 0:
        print(f"Mean p(rain) from 11:30 : {after['rain_probability'].mean():.3f}")
    if len(before) > 0:
        print(f"Mean p(rain) before 11:30 : {before['rain_probability'].mean():.3f}")


### 3.3 Detection Timeline

Overview of all detected rain events across the full recording day.
The green dashed line marks the logged rain onset at 11:30 UTC.


In [ ]:
if not df.empty:
    base_dt = datetime(2026, 5, 13, 8, 30, 0)
    fig, ax = plt.subplots(figsize=(16, 3.5))

    for _, row in df.iterrows():
        try:
            t0 = datetime.strptime(f"2026-05-13 {row['utc_start']}", "%Y-%m-%d %H:%M:%S")
            t1 = datetime.strptime(f"2026-05-13 {row['utc_end']}",   "%Y-%m-%d %H:%M:%S")
            x0 = (t0 - base_dt).seconds / 3600
            w  = max((t1 - t0).seconds / 3600, 0.015)
            c  = plt.cm.Reds(0.25 + 0.75 * row["rain_probability"])
            ax.barh(0, w, left=x0, height=0.5, color=c,
                    edgecolor="darkred", lw=0.4, alpha=0.9)
        except Exception:
            pass

    known_events = {
        "09:32": "Ducks",     "10:07": "Ducks",
        "10:37": "Otter USV", "10:51": "Hydro OUT",
        "11:11": "Hydro IN",  "11:30": "Rain onset",
        "12:45": "Otter USV", "14:17": "Otter USV", "14:53": "Otter USV",
    }
    for ts, event in known_events.items():
        try:
            t = datetime.strptime(f"2026-05-13 {ts}:00", "%Y-%m-%d %H:%M:%S")
            x = (t - base_dt).seconds / 3600
            if "Rain" in event:
                ax.axvline(x, color="green", ls="--", lw=2.5, zorder=5)
                ax.text(x + 0.03, 0.28, f"{ts} {event}",
                        fontsize=8, color="green", fontweight="bold")
            elif "Duck" in event:
                ax.axvline(x, color="saddlebrown", ls=":", lw=1.2, alpha=0.5)
            elif "Otter" in event:
                ax.axvline(x, color="steelblue",   ls=":", lw=1.2, alpha=0.5)
            elif "OUT" in event or "IN" in event:
                ax.axvline(x, color="orange",       ls="-.", lw=1.2, alpha=0.6)
        except Exception:
            pass

    ax.set_xlim(0, 6.6)
    xticks = np.arange(0, 6.6, 0.5)
    ax.set_xticks(xticks)
    ax.set_xticklabels(
        [(base_dt + timedelta(hours=float(h))).strftime("%H:%M") for h in xticks],
        fontsize=8)
    ax.set_xlabel("UTC Time", fontsize=10)
    ax.set_title(
        "Figure 4: Detected rain events — 2026-05-13
"
        "Red bars = detected rain (darker = higher probability)  |  "
        "Green = logged onset  |  Blue = Otter  |  Brown = Ducks",
        fontweight="bold")
    ax.set_yticks([])
    plt.tight_layout()
    plt.show()


### 3.4 CSV Export

In [ ]:
if not df.empty:
    cols = ["file_name", "utc_start", "utc_end",
            "duration_s", "mean_energy_db", "lf_energy_db", "rain_probability"]
    df[cols].to_csv(OUTPUT_CSV, index=False)
    print(f"Saved: {OUTPUT_CSV}")
    print(f"  {len(df)} segments  |  {df['file_name'].nunique()} files")
    print()
    print(df[cols].to_string(index=False))
else:
    print("No results to save.")


## 4. Discussion

### 4.1 Methodology Notes

The detection pipeline operates on the raw signal rather than the denoised version.
Spectral gating with `prop_decrease=0.999` — used in Section 2.3 for audio enhancement
— suppresses stationary broadband noise, which is precisely the acoustic character of
rainfall. Applying it before detection would remove the target signal. The denoised
signal is therefore used only for visualization and listening.

The Clip-Median threshold addresses a recurring issue in this dataset: the Otter USV
produces short energy spikes that bias standard mean-based thresholds upward, causing
genuine rain events to fall below the detection boundary. Clipping at the 60th percentile
before computing the median isolates the file's quiet background level and makes the
threshold robust to these outliers.

### 4.2 Limitations

- The adaptive threshold is computed per file. A file entirely dominated by rain will
  have an elevated baseline, potentially causing the detector to under-report continuous
  rain events.
- Parameter values were calibrated against this specific dataset and deployment.
  Applying the detector to different environments or sensor configurations would require
  re-calibration.
- Validation is constrained by a single ground-truth timestamp (11:30 UTC). Frame-level
  evaluation would require manual annotation of the recordings.

### 4.3 Relation to Group Report Event Table

The group report (Table 2) identifies the 11:30 UTC file as "Otter driving sporadically
and continuously" with certainty 1, and does not list rain as a detected event. The
automated detector identifies sustained broadband HF energy from 11:30 UTC onward,
consistent with the logged rain onset. The LF-HF ratio panel (Panel 5) confirms that
post-11:30 detections are characterized by LF ≈ HF — the broadband signature of rain —
as opposed to the LF-dominant signature of the Otter USV recorded at other times.

---

### References

Nystuen, J. A., & Medwin, H. (1993). Underwater sound produced by rainfall: Secondary
splashes of aerosols. *Journal of the Acoustical Society of America*, 94(2), 1005–1018.

Dannemann, A., Haider, B., & Schallück, J. (2026). *IMS Sensor Processing Project —
Hydrophone*. University of Bremen.
